# Confluence — FULL-SCALE runs with Google Drive checkpoints

**Before you start:** `Runtime -> Change runtime type -> T4 GPU`, then `Runtime -> Run all`.

What this does:
- Runs every paper experiment at full scale (more seeds, epochs, data than the paper draft).
- **Checkpoints to your Google Drive after every single run** (`MyDrive/Confluence_runs/fullscale_v1/`).
- **Resumable:** if Colab disconnects, just `Run all` again. Finished runs are skipped, so you lose at most the one run in progress.
- Ends with summary tables (mean ± std) saved to Drive, ready to paste into the paper.

Rough time on a T4: headline ~30-60 min, transformer sweep ~30-60 min, synthetic suite ~40 min.
Keep the tab open (free Colab disconnects idle sessions; Colab Pro is steadier).

## 1. Mount Google Drive (checkpoint location)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import os
RUN_DIR = '/content/drive/MyDrive/Confluence_runs/fullscale_v1'   # same folder every time -> resume works
os.makedirs(f'{RUN_DIR}/logs', exist_ok=True)
print('checkpoints ->', RUN_DIR)
print('already finished:', sorted(f[:-5] for f in os.listdir(f'{RUN_DIR}/logs') if f.endswith('.done')))

## 2. Get the code (main branch) + GPU check
The repo is **private**, so Colab needs a GitHub token to clone it. Either:
- add a Colab secret named `GITHUB_TOKEN` (key icon in the left sidebar, turn on notebook access), or
- paste the token when this cell asks.

Make the token at github.com/settings/tokens: **Fine-grained token**, repository access = only `Confluence`, permission **Contents: Read-only**.
The token stays in memory. It is never written to Drive or the repo.

In [ ]:
import os, subprocess, shutil
REPO = '/content/Confluence'
URL = 'github.com/tanushappapogu-max/Confluence.git'
os.environ['GIT_TERMINAL_PROMPT'] = '0'   # fail fast instead of hanging on a password prompt

def get_token():
    try:
        from google.colab import userdata
        t = userdata.get('GITHUB_TOKEN')
        if t: return t.strip()
    except Exception:
        pass
    from getpass import getpass
    return getpass('Repo is private. Paste a GitHub token (Contents: read-only): ').strip()

def git(*args):
    return subprocess.run(['git', *args], capture_output=True, text=True)

if not os.path.exists(f'{REPO}/moe_transformer.py'):
    shutil.rmtree(REPO, ignore_errors=True)
    if git('clone', '-q', f'https://{URL}', REPO).returncode != 0:
        TOKEN = get_token()
        r = git('clone', '-q', f'https://{TOKEN}@{URL}', REPO)
        if r.returncode != 0:
            raise RuntimeError('Clone failed. Check the token has read access to Confluence.\n' + r.stderr.replace(TOKEN, '***'))
        git('-C', REPO, 'remote', 'set-url', 'origin', f'https://{URL}')   # do not keep the token in .git/config
else:
    if git('-C', REPO, 'pull', '-q', 'origin', 'main').returncode != 0:
        TOKEN = globals().get('TOKEN') or get_token()
        git('-C', REPO, 'pull', '-q', f'https://{TOKEN}@{URL}', 'main')

os.chdir(REPO)
assert os.path.exists('moe_transformer.py'), 'code not found - clone failed'
print('code at', os.getcwd(), '|', git('log', '-1', '--oneline').stdout.strip())
!pip -q install torch numpy matplotlib
import torch
print('CUDA:', torch.cuda.is_available(), '|', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU ONLY - switch runtime to GPU')

## 3. Checkpointing helpers (run each experiment, stream its log to Drive, mark done)

In [ ]:
import subprocess, time, re, json, statistics as st

def run(name, cmd, env_extra=None, strict=True):
    '''Run a command in the repo, streaming output live AND to a Drive log. Skips if already checkpointed.
    strict=True stops the notebook on a crash instead of silently burning through seeds.'''
    log, done = f'{RUN_DIR}/logs/{name}.log', f'{RUN_DIR}/logs/{name}.done'
    if os.path.exists(done):
        print(f'[skip] {name} (checkpointed)'); return open(log).read()
    env = dict(os.environ, PYTHONUNBUFFERED='1', **(env_extra or {}))
    print(f'[run ] {name}'); t0 = time.time()
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, shell=True, env=env, stdout=subprocess.PIPE,
                             stderr=subprocess.STDOUT, text=True, bufsize=1, cwd=REPO)
        for line in p.stdout:
            f.write(line); f.flush()
            if any(k in line for k in ('gate', 'flow (ours)', 'dense', 'seed', 'epoch', 'total', 'Error', 'Traceback')):
                print('   ', line, end='')
        p.wait()
    if p.returncode == 0:
        open(done, 'w').write(f'{time.time()-t0:.0f}s')
        print(f'[done] {name} in {time.time()-t0:.0f}s -> saved to Drive')
    else:
        text = open(log).read()
        print(f'[FAIL] {name} exit {p.returncode}; log kept on Drive, will retry on next Run all')
        print('    last lines of log:\n    ' + '\n    '.join(text.strip().splitlines()[-15:]))
        if strict: raise RuntimeError(f'{name} crashed - fix the error above, then Run all again (finished runs are kept)')
    return open(log).read()

ROW = re.compile(r'^(dense \(all\)|top-1 gate|top-2 gate|flow \(ours\))\s+([\d.]+)\s+([\d.]+)\s+(\S+)', re.M)
def parse(text):
    if 'too few legal paths' in text: return None
    out = {m.group(1): dict(acc=float(m.group(2)), active=float(m.group(3)),
                            illegal=None if m.group(4) == 'n/a' else float(m.group(4)))
           for m in ROW.finditer(text)}
    return out or None

def summarize(results, title):
    '''results: list of per-seed dicts -> mean±std table, printed and saved to Drive.'''
    if not results: print(f'## {title}: no valid seeds, nothing to summarize'); return ''
    routers = ['dense (all)', 'top-1 gate', 'top-2 gate', 'flow (ours)']
    lines = [f'## {title}  (n={len(results)} seeds)', '', '| router | token acc | active/layer | illegal |', '|---|---|---|---|']
    def ms(v): return f'{st.mean(v):.3f} ± {st.pstdev(v):.3f}' if len(v) > 1 else f'{v[0]:.3f}'
    for r in routers:
        rows = [x[r] for x in results if r in x]
        if not rows: continue
        ill = [x['illegal'] for x in rows if x['illegal'] is not None]
        lines.append(f"| {r} | {ms([x['acc'] for x in rows])} | {ms([x['active'] for x in rows])} | {ms(ill) if ill else 'n/a'} |")
    txt = '\n'.join(lines); print(txt)
    with open(f'{RUN_DIR}/SUMMARY.md', 'a') as f: f.write(txt + '\n\n')
    return txt
print('helpers ready')

## 4. HEADLINE — MoE transformer, 10 seeds, full scale
Flow router vs top-k gate at **equal compute**. Each seed is its own checkpoint. Some random seeds draw a
graph with too few legal paths; those are skipped and **recorded**, and the loop keeps going until it has 10 valid seeds.

In [ ]:
HEAD = dict(DENSITY='0.5', EPOCHS='20', NTRAIN='30000', NTEST='6000', NSEEDS='1')
valid, skipped, seed = [], [], 0
while len(valid) < 10 and seed < 40:
    r = parse(run(f'transformer_d0.5_seed{seed}', 'python3 moe_transformer.py', dict(HEAD, SEED_START=str(seed))))
    (valid.append(r) if r else skipped.append(seed)); seed += 1
print('valid seeds:', len(valid), '| skipped (too few legal paths):', skipped)
json.dump({'valid': valid, 'skipped': skipped}, open(f'{RUN_DIR}/headline_raw.json', 'w'), indent=1)
summarize(valid, 'HEADLINE: MoE transformer, density 0.5, 20 epochs, 30k train')

## 5. Transformer sweep across legality densities (5 valid seeds each)

In [ ]:
sweep = {}
for den in ['0.4', '0.5', '0.6', '0.8', '1.0']:
    vals, s = [], 0
    while len(vals) < 5 and s < 25:
        r = parse(run(f'sweep_d{den}_seed{s}', 'python3 moe_transformer.py',
                      dict(DENSITY=den, EPOCHS='15', NTRAIN='20000', NTEST='4000', NSEEDS='1', SEED_START=str(s))))
        if r: vals.append(r)
        s += 1
    sweep[den] = vals
    if vals: summarize(vals, f'SWEEP: density {den}')
json.dump(sweep, open(f'{RUN_DIR}/sweep_raw.json', 'w'), indent=1)

## 6. Synthetic suite: coupling crossover, MoE routing (4 seeds), FFN frontier, implicit-diff check

In [ ]:
run('synthetic_density_sweep', 'python3 density_sweep.py', strict=False)
run('synthetic_moe_routing', 'python3 mycelial_final.py', strict=False)
run('synthetic_ffn_frontier', 'python3 moe_layer_demo.py', strict=False)
run('implicit_diff_check', 'python3 implicit_diff.py', strict=False)

## 7. (Optional, slow) MetaQA real-data reproduction
Reproduces the MetaQA table yourself: the no-flow ablation (~1-2 h) and the flow model
(**~5-6 h**, CPU-bound, per-query solves). Checkpointed, so you can stop and resume.

In [ ]:
if not os.path.exists('data/metaqa/kb/kb.txt'):
    !git clone -q --depth 1 https://github.com/Tiny-Small/CS5284_Project /tmp/mqa
    import glob, shutil
    def stage(pats, dst, big=False):
        hits = [h for p in pats for h in glob.glob(f'/tmp/mqa/**/{p}', recursive=True) if os.path.getsize(h) > 1000]
        if not hits: return None
        src = max(hits, key=os.path.getsize) if big else min(hits, key=len)
        os.makedirs(os.path.dirname(dst), exist_ok=True); shutil.copy(src, dst); return src
    stage(['kb.txt'], 'data/metaqa/kb/kb.txt', big=True)
    for h in ['2', '3']:
        for sp in ['train', 'test']:
            stage([f'{h}-hop/qa_{sp}.txt', f'*{h}-hop*/qa_{sp}.txt'], f'data/metaqa/{h}-hop/qa_{sp}.txt')
!wc -l data/metaqa/kb/kb.txt data/metaqa/2-hop/qa_train.txt
run('metaqa_2hop_noflow', 'python3 metaqa_confluence_2hop_ablation.py', strict=False)
run('metaqa_2hop_flow_v2', 'python3 metaqa_confluence_2hop_v2.py', strict=False)   # slowest; fine to leave for last

## 8. Figures + everything copied to Drive

In [ ]:
os.chdir(REPO)
for s in ['make_fig_coupling.py', 'make_fig_moe.py', 'make_fig_implicit.py']:
    r = subprocess.run(['python3', s], capture_output=True, text=True, cwd=REPO)
    print(s, 'ok' if r.returncode == 0 else 'FAILED:\n' + r.stderr[-800:])
os.makedirs(f'{RUN_DIR}/figs', exist_ok=True)
from IPython.display import Image, display
for f in ['coupling', 'moe_frontier', 'implicit_diff']:
    p = f'{REPO}/paper/figs/{f}.png'
    if os.path.exists(p):
        shutil.copy(p, f'{RUN_DIR}/figs/'); display(Image(filename=p))
    else:
        print('missing figure:', p)
print(open(f'{RUN_DIR}/SUMMARY.md').read() if os.path.exists(f'{RUN_DIR}/SUMMARY.md') else 'no summary yet')
print('\nAll logs, raw JSON, summary tables and figures are in:', RUN_DIR)